# LeetCode 1091 — Shortest Path in Binary Matrix bằng Python

Link đề: https://leetcode.com/problems/shortest-path-in-binary-matrix/ · Độ khó: **Medium** · Chủ đề: BFS (duyệt theo chiều rộng), Graph (đồ thị), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: cho ma trận nhị phân `n x n` `grid`. Trả về **độ dài đường đi "sạch" ngắn nhất** từ ô trên-trái `(0, 0)` đến ô dưới-phải `(n - 1, n - 1)`, hoặc `-1` nếu không có. Đường đi sạch: mọi ô đi qua đều là `0`, hai ô liên tiếp kề nhau theo **8 hướng** (chung cạnh hoặc chung đỉnh, tức cả đường chéo); **độ dài** là **số ô** đã đi qua (kể cả ô đầu và ô cuối). Ý tưởng chính: mọi bước đều có chi phí `1`, nên **BFS** từ `(0, 0)` cho đường đi ngắn nhất; nhớ kiểm tra ô đầu và ô cuối phải bằng `0`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận là một **đồ thị** (mỗi ô `0` là một đỉnh, nối với tối đa `8` ô lân cận cũng là `0`), độ dài đường đi tính bằng **số ô**.
2. Nắm các công cụ Python cần thiết: `collections.deque`, tuple tọa độ `(r, c, d)`, vòng lặp `for dr in (-1, 0, 1)`, mảng `seen` hai chiều, `heapq`.
3. Hiểu **BFS tìm đường ngắn nhất trên đồ thị không trọng số**: đánh dấu đã thăm **ngay khi đưa vào hàng đợi**, không phải khi lấy ra.
4. Đi qua 4 cách: liệt kê mọi đường đi đơn (brute force, dùng làm oracle cho lưới rất nhỏ), BFS chuẩn (cách nộp chính), BFS hai chiều (từ hai đầu gặp nhau) và A* với heuristic Chebyshev.
5. Tránh các lỗi kinh điển: quên kiểm tra ô đầu/ô cuối là `1`, chỉ đi `4` hướng, đếm số cạnh thay vì số ô, dùng DFS (không đảm bảo ngắn nhất), kiểm tra đích sai chỗ làm hỏng trường hợp `n = 1`, và đánh dấu đã thăm muộn làm hàng đợi phình to.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `n x n` `grid`, trả về **độ dài của đường đi sạch ngắn nhất** trong ma trận. Nếu không có đường đi sạch, trả về `-1`.

Một **đường đi sạch** trong ma trận nhị phân là đường đi từ ô **trên-trái** `(0, 0)` đến ô **dưới-phải** `(n - 1, n - 1)` sao cho:

- Mọi ô trên đường đi đều là `0`.
- Hai ô liên tiếp trên đường đi **kề nhau theo 8 hướng** (hai ô khác nhau và chung cạnh hoặc chung đỉnh).

**Độ dài** của một đường đi sạch là **số ô** nó đi qua.

**Ví dụ 1:** `grid = [[0,1],[1,0]]` -> `2` (đi chéo từ `(0, 0)` sang `(1, 1)`).

**Ví dụ 2:** `grid = [[0,0,0],[1,1,0],[1,1,0]]` -> `4`.

**Ví dụ 3:** `grid = [[1,0,0],[1,1,0],[1,1,0]]` -> `-1` (ô đầu là `1`).

**Ràng buộc:**

- `n == grid.length == grid[i].length`
- `1 <= n <= 100`
- `grid[i][j]` là `0` hoặc `1`.

In [1]:
from typing import List, Optional
from collections import deque
import heapq, random, time
from itertools import product

examples = [
    ([[0, 1], [1, 0]], 2),
    ([[0, 0, 0], [1, 1, 0], [1, 1, 0]], 4),
    ([[1, 0, 0], [1, 1, 0], [1, 1, 0]], -1),
]
for grid, expected in examples:
    print(f"grid={grid} -> kỳ vọng {expected}")

grid=[[0, 1], [1, 0]] -> kỳ vọng 2
grid=[[0, 0, 0], [1, 1, 0], [1, 1, 0]] -> kỳ vọng 4
grid=[[1, 0, 0], [1, 1, 0], [1, 1, 0]] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Ma trận như đồ thị và 8 hướng đi

Mỗi ô `(r, c)` có tối đa `8` ô lân cận `(r + dr, c + dc)` với `dr, dc` thuộc `{-1, 0, 1}` và không đồng thời bằng `0`. Chỉ đi vào ô trong ma trận (`0 <= nr < n`, `0 <= nc < n`) và có giá trị `0`.

In [2]:
dirs = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
print(len(dirs), dirs)
n, r, c = 3, 0, 0
print("lân cận hợp lệ của (0, 0) trong lưới 3 x 3:", [(r + dr, c + dc) for dr, dc in dirs if 0 <= r + dr < n and 0 <= c + dc < n])

8 [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
lân cận hợp lệ của (0, 0) trong lưới 3 x 3: [(0, 1), (1, 0), (1, 1)]


### 3.2. Hàng đợi `deque` và BFS

BFS dùng hàng đợi (vào sau ra sau): các ô cách điểm xuất phát `d` bước được lấy ra **trước** mọi ô cách `d + 1` bước. `collections.deque` có `append` và `popleft` đều `O(1)`.

In [3]:
q = deque([(0, 0, 1)])
q.append((0, 1, 2))
q.append((1, 1, 2))
print(q.popleft(), q.popleft(), list(q))

(0, 0, 1) (0, 1, 2) [(1, 1, 2)]


### 3.3. Đường đi ngắn nhất: BFS và "đánh dấu khi đưa vào"

Trên đồ thị không trọng số, lần đầu BFS **chạm tới** một ô chính là đường ngắn nhất tới ô đó. Vì vậy ta đánh dấu ô là "đã thăm" **ngay khi đưa vào hàng đợi** để mỗi ô vào hàng đợi **đúng một lần**; đánh dấu muộn (khi lấy ra) làm một ô bị đưa vào nhiều lần.

In [4]:
def count_pushes_early(n):
    """Đếm số lần đưa ô vào hàng đợi khi đánh dấu ngay lúc đưa vào (lưới n x n toàn 0)."""
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    q, pushes = deque([(0, 0)]), 1
    while q:
        r, c = q.popleft()
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not seen[nr][nc]:
                seen[nr][nc] = True
                q.append((nr, nc))
                pushes += 1
    return pushes

print("đánh dấu khi đưa vào, lưới 4 x 4:", count_pushes_early(4), "lần đưa vào hàng đợi (= số ô)")
assert count_pushes_early(4) == 16

đánh dấu khi đưa vào, lưới 4 x 4: 16 lần đưa vào hàng đợi (= số ô)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `grid = [[0,0,0],[1,1,0],[1,1,0]]`.

- Bắt đầu tại `(0, 0)` với độ dài `1`.
- Từ `(0, 0)`: các ô lân cận hợp lệ (giá trị `0`): `(0, 1)`. (Ô `(1, 0)` và `(1, 1)` là `1`.) ⇒ `(0, 1)` độ dài `2`.
- Từ `(0, 1)`: lân cận `(0, 2)` và `(1, 2)` (đường chéo, giá trị `0`) ⇒ độ dài `3`.
- Từ `(0, 2)` hoặc `(1, 2)`: có `(2, 2)` kề `(1, 2)` ⇒ độ dài `4`; đây là ô cuối ⇒ kết quả `4` ✓.

Ví dụ 1: `[[0,1],[1,0]]`: từ `(0, 0)` đi **chéo** sang `(1, 1)`: độ dài `2` ✓ (chỉ đi `4` hướng sẽ không đến được).

Ví dụ 3: ô đầu `(0, 0)` là `1` ⇒ không thể bắt đầu ⇒ `-1`.

## 5. Cách 1 — Brute Force: liệt kê mọi đường đi đơn

**Ý tưởng.** Dùng quay lui (DFS) liệt kê **mọi đường đi đơn** (không đi lại ô đã qua) từ `(0, 0)`; mỗi khi tới `(n - 1, n - 1)` thì ghi nhận số ô đã đi; lấy nhỏ nhất. Đúng định nghĩa đề, không dùng BFS nên là **oracle độc lập** (chỉ dùng được cho lưới rất nhỏ, `n <= 4`).

In [5]:
def shortest_brute(grid: List[List[int]]) -> int:
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    best = [float("inf")]
    on_path = [[False] * n for _ in range(n)]

    def dfs(r, c, length):
        if length >= best[0]:                       # cắt tỉa đơn giản: không thể tốt hơn
            return
        if (r, c) == (n - 1, n - 1):
            best[0] = length
            return
        on_path[r][c] = True
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not on_path[nr][nc]:
                dfs(nr, nc, length + 1)
        on_path[r][c] = False

    dfs(0, 0, 1)
    return -1 if best[0] == float("inf") else best[0]

for grid, expected in examples:
    got = shortest_brute(grid)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = 2 OK
brute = 4 OK
brute = -1 OK


**Độ phức tạp và nút thắt.** Số đường đi đơn tăng cực nhanh theo kích thước (lưới toàn `0`: số lần gọi tăng hàng trăm lần mỗi khi `n` tăng thêm `1`). Đo thử số lần gọi và thời gian trên lưới toàn `0`:

In [6]:
def brute_nodes(n):
    """Đếm số lần gọi dfs không cắt tỉa trên lưới n x n toàn 0 (liệt kê mọi đường đi đơn)."""
    grid = [[0] * n for _ in range(n)]
    count = [0]
    on_path = [[False] * n for _ in range(n)]
    def dfs(r, c):
        count[0] += 1
        if (r, c) == (n - 1, n - 1):
            return
        on_path[r][c] = True
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not on_path[nr][nc]:
                dfs(nr, nc)
        on_path[r][c] = False
    dfs(0, 0)
    return count[0]

for n in (2, 3, 4):
    t0 = time.perf_counter()
    k = brute_nodes(n)
    dt = time.perf_counter() - t0
    print(f"lưới {n}x{n} toàn 0: {k:9} lời gọi dfs, {dt * 1000:8.1f} ms")

lưới 2x2 toàn 0:        10 lời gọi dfs,      0.0 ms
lưới 3x3 toàn 0:       827 lời gọi dfs,      0.9 ms


lưới 4x4 toàn 0:    551576 lời gọi dfs,    536.9 ms


Nút thắt: ta xét **mọi** đường đi trong khi chỉ cần đường **ngắn nhất**. Mỗi bước có chi phí bằng nhau nên có thể mở rộng **theo từng lớp khoảng cách** (BFS).

## 6. Ý tưởng thuật toán / bước đột phá

**BFS theo lớp khoảng cách.** Đặt `(0, 0)` ở lớp `1` (độ dài `1`). Lớp `d + 1` gồm các ô `0` chưa thăm kề với một ô ở lớp `d`. Lần đầu tới được `(n - 1, n - 1)` ở lớp `d` thì `d` là độ dài ngắn nhất (mọi đường đi tới ô đó phải đi qua các lớp `1, 2, ..., d` lần lượt, vì mỗi bước tăng độ dài đúng `1`).

Lưu ý:

- Nếu `grid[0][0] == 1` hoặc `grid[n - 1][n - 1] == 1` thì `-1`.
- `n = 1` và ô duy nhất là `0` thì đáp án `1` (ô đầu cũng là ô cuối).
- Đánh dấu thăm **khi đưa vào hàng đợi**.

| Cách | Ý tưởng |
|---|---|
| 2 | BFS một chiều, lưu độ dài trong hàng đợi (nộp) |
| 3 | BFS hai chiều: mở rộng cả từ đầu lẫn từ cuối, gặp nhau ở giữa |
| 4 | A* với heuristic là khoảng cách Chebyshev tới đích |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS một chiều (cách nộp chính)

**Giải thích từng đoạn.**

- Nếu ô đầu hoặc ô cuối là `1`: `-1`.
- `queue` chứa `(r, c, d)` với `d` là số ô của đường đi tới `(r, c)`; `seen` đánh dấu đã đưa vào hàng đợi.
- Lấy `(r, c, d)`; nếu là ô cuối thì trả về `d`.
- Với mỗi trong `8` hướng: nếu ô lân cận nằm trong lưới, bằng `0` và chưa thăm thì đánh dấu và đưa `(nr, nc, d + 1)` vào hàng đợi.
- Hết hàng đợi mà chưa tới: `-1`.

In [7]:
def shortest_bfs(grid: List[List[int]]) -> int:
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
    return -1

for grid, expected in examples:
    got = shortest_bfs(grid)
    print(f"BFS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

BFS = 2 OK
BFS = 4 OK
BFS = -1 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:                 # ô đầu hoặc ô cuối bị chặn
            return -1
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        queue = deque([(0, 0, 1)])                           # (hàng, cột, số ô đã đi)
        while queue:
            r, c, d = queue.popleft()
            if r == n - 1 and c == n - 1:                    # tới ô cuối: BFS đảm bảo d nhỏ nhất
                return d
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):                        # 8 hướng
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                        seen[nr][nc] = True                  # đánh dấu ngay khi đưa vào hàng đợi
                        queue.append((nr, nc, d + 1))
        return -1

sol = lambda g: Solution().shortestPathBinaryMatrix([row[:] for row in g])
for grid, expected in examples:
    assert sol(grid) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** mỗi ô vào hàng đợi tối đa một lần và xét `8` lân cận ⇒ thời gian `O(n^2)`, bộ nhớ `O(n^2)`.

---

### 7.2. Cách 3 — BFS hai chiều

**Ý tưởng khác.** Mở rộng đồng thời từ ô đầu (khoảng cách `1`) và từ ô cuối (khoảng cách `1`). Mỗi vòng, mở rộng **trọn một lớp** của phía có biên nhỏ hơn. Nếu một ô lân cận đã được phía kia thăm thì ta tìm được một đường đi có số ô bằng `dist[u] + dist_other[v]` (với `u` là ô đang mở rộng, `v` là ô lân cận đã thăm phía kia). Do trong cùng một lớp có thể thấy nhiều điểm gặp nhau, ta lấy **nhỏ nhất** trong cả lớp rồi trả về. BFS hai chiều thăm ít ô hơn khi đồ thị dày.

In [9]:
def shortest_bidirectional(grid: List[List[int]]) -> int:
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    if n == 1:
        return 1
    dist_a, dist_b = {(0, 0): 1}, {(n - 1, n - 1): 1}
    front_a, front_b = [(0, 0)], [(n - 1, n - 1)]
    while front_a and front_b:
        if len(front_a) > len(front_b):                       # luôn mở rộng phía có biên nhỏ hơn
            front_a, front_b = front_b, front_a
            dist_a, dist_b = dist_b, dist_a
        best = None
        nxt = []
        for r, c in front_a:
            for dr, dc in dirs:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < n and 0 <= nc < n) or grid[nr][nc]:
                    continue
                if (nr, nc) in dist_b:                       # gặp phía kia: đường qua cạnh (r, c) - (nr, nc)
                    cand = dist_a[(r, c)] + dist_b[(nr, nc)]
                    best = cand if best is None else min(best, cand)
                elif (nr, nc) not in dist_a:
                    dist_a[(nr, nc)] = dist_a[(r, c)] + 1
                    nxt.append((nr, nc))
        if best is not None:                                  # đã duyệt trọn lớp: lấy nhỏ nhất
            return best
        front_a = nxt
    return -1

for grid, expected in examples:
    got = shortest_bidirectional(grid)
    print(f"BFS hai chiều = {got}", "OK" if got == expected else "SAI")
    assert got == expected

BFS hai chiều = 2 OK
BFS hai chiều = 4 OK
BFS hai chiều = -1 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:
            return -1
        if n == 1:
            return 1
        dist_a, dist_b = {(0, 0): 1}, {(n - 1, n - 1): 1}        # khoảng cách (số ô) từ mỗi đầu
        front_a, front_b = [(0, 0)], [(n - 1, n - 1)]            # biên hiện tại của mỗi phía
        while front_a and front_b:
            if len(front_a) > len(front_b):                      # mở rộng phía có biên nhỏ hơn
                front_a, front_b = front_b, front_a
                dist_a, dist_b = dist_b, dist_a
            best = None
            nxt = []
            for r, c in front_a:
                for dr in (-1, 0, 1):
                    for dc in (-1, 0, 1):
                        nr, nc = r + dr, c + dc
                        if (dr or dc) and 0 <= nr < n and 0 <= nc < n and not grid[nr][nc]:
                            if (nr, nc) in dist_b:               # hai phía gặp nhau
                                cand = dist_a[(r, c)] + dist_b[(nr, nc)]
                                best = cand if best is None else min(best, cand)
                            elif (nr, nc) not in dist_a:
                                dist_a[(nr, nc)] = dist_a[(r, c)] + 1
                                nxt.append((nr, nc))
            if best is not None:                                 # mở rộng trọn lớp rồi mới trả về nhỏ nhất
                return best
            front_a = nxt
        return -1

sol_bi = lambda g: Solution().shortestPathBinaryMatrix([row[:] for row in g])
for grid, expected in examples:
    assert sol_bi(grid) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** trường hợp xấu `O(n^2)` thời gian và bộ nhớ (như BFS một chiều); trên lưới thoáng thường thăm ít ô hơn.

---

### 7.3. Cách 4 — A* với heuristic Chebyshev

**Ý tưởng khác.** A* ưu tiên ô có `f = g + h` nhỏ nhất, với `g` là số ô đã đi và `h` là **ước lượng số ô còn lại** tới đích. Với di chuyển `8` hướng, khoảng cách `Chebyshev` `max(|n - 1 - r|, |n - 1 - c|)` là số bước tối thiểu (không bao giờ ước lượng thừa, và nhất quán), nên A* vẫn trả về đường ngắn nhất.

In [11]:
def shortest_astar(grid: List[List[int]]) -> int:
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    h = lambda r, c: max(n - 1 - r, n - 1 - c)             # số bước tối thiểu còn lại (Chebyshev)
    best_g = {(0, 0): 1}
    heap = [(1 + h(0, 0), 1, 0, 0)]                        # (f, g, r, c)
    while heap:
        f, g, r, c = heapq.heappop(heap)
        if g > best_g.get((r, c), float("inf")):           # bản cũ, đã có đường tốt hơn
            continue
        if (r, c) == (n - 1, n - 1):
            return g
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and g + 1 < best_g.get((nr, nc), float("inf")):
                best_g[(nr, nc)] = g + 1
                heapq.heappush(heap, (g + 1 + h(nr, nc), g + 1, nr, nc))
    return -1

for grid, expected in examples:
    got = shortest_astar(grid)
    print(f"A* = {got}", "OK" if got == expected else "SAI")
    assert got == expected

A* = 2 OK
A* = 4 OK
A* = -1 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        import heapq
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:
            return -1
        best_g = {(0, 0): 1}                                     # số ô nhỏ nhất đã biết tới mỗi ô
        heap = [(max(n - 1, n - 1) + 1, 1, 0, 0)]                # (f = g + h, g, r, c), h = khoảng cách Chebyshev tới đích
        while heap:
            f, g, r, c = heapq.heappop(heap)
            if g > best_g[(r, c)]:                               # phần tử cũ
                continue
            if r == n - 1 and c == n - 1:
                return g
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if (dr or dc) and 0 <= nr < n and 0 <= nc < n and not grid[nr][nc]:
                        ng = g + 1
                        if ng < best_g.get((nr, nc), float("inf")):
                            best_g[(nr, nc)] = ng
                            h = max(n - 1 - nr, n - 1 - nc)
                            heapq.heappush(heap, (ng + h, ng, nr, nc))
        return -1

sol_astar = lambda g: Solution().shortestPathBinaryMatrix([row[:] for row in g])
for grid, expected in examples:
    assert sol_astar(grid) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** trường hợp xấu `O(n^2 log n)`; trên lưới thoáng A* thăm rất ít ô (đi gần đường chéo).

## 8. TRACE từng bước

### 8.1. BFS cho ví dụ 2

In từng lần lấy ô ra khỏi hàng đợi và các ô mới được đưa vào.

In [13]:
def trace_bfs(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        print("ô đầu hoặc ô cuối là 1 -> -1")
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        print(f"lấy ({r}, {c}) độ dài {d}", end="")
        if r == n - 1 and c == n - 1:
            print(" -> là ô cuối! trả về", d)
            return d
        added = []
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
                added.append((nr, nc))
        print(" -> thêm", added)
    print("hết hàng đợi -> -1")
    return -1

assert trace_bfs([[0, 0, 0], [1, 1, 0], [1, 1, 0]]) == 4
print()
assert trace_bfs([[0, 1], [1, 0]]) == 2
print()
assert trace_bfs([[1, 0, 0], [1, 1, 0], [1, 1, 0]]) == -1

lấy (0, 0) độ dài 1 -> thêm [(0, 1)]
lấy (0, 1) độ dài 2 -> thêm [(0, 2), (1, 2)]
lấy (0, 2) độ dài 3 -> thêm []
lấy (1, 2) độ dài 3 -> thêm [(2, 2)]
lấy (2, 2) độ dài 4 -> là ô cuối! trả về 4

lấy (0, 0) độ dài 1 -> thêm [(1, 1)]
lấy (1, 1) độ dài 2 -> là ô cuối! trả về 2

ô đầu hoặc ô cuối là 1 -> -1


### 8.2. Các lớp BFS (lớp = số ô) trên lưới 4 x 4

In ma trận khoảng cách để thấy các lớp lan rộng theo đường chéo.

In [14]:
grid = [[0, 0, 1, 0], [1, 0, 1, 0], [0, 0, 0, 1], [0, 1, 0, 0]]
n = len(grid)
dist = [[0] * n for _ in range(n)]
dist[0][0] = 1
queue = deque([(0, 0)])
while queue:
    r, c = queue.popleft()
    for dr, dc in dirs:
        nr, nc = r + dr, c + dc
        if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not dist[nr][nc]:
            dist[nr][nc] = dist[r][c] + 1
            queue.append((nr, nc))
print("ma trận khoảng cách (0 = chưa tới hoặc ô bị chặn):")
for row, grow in zip(dist, grid):
    print("  ", " ".join(f"{x:2}" if not g else " #" for x, g in zip(row, grow)))
assert dist[n - 1][n - 1] == shortest_bfs(grid)
print("đáp án =", dist[n - 1][n - 1])

ma trận khoảng cách (0 = chưa tới hoặc ô bị chặn):
    1  2  #  5
    #  2  #  4
    3  3  3  #
    4  #  4  4
đáp án = 4


### 8.3. Vì sao phải đánh dấu khi đưa vào hàng đợi

Nếu đánh dấu muộn (khi lấy ra), cùng một ô bị đưa vào hàng đợi nhiều lần. Ta đếm số lần đưa vào trên lưới toàn `0`.

In [15]:
def count_pushes_late(n, cap=3 * 10 ** 5):
    grid = [[0] * n for _ in range(n)]
    seen = [[False] * n for _ in range(n)]
    queue = deque([(0, 0, 1)])
    pushes = 1
    while queue and pushes < cap:
        r, c, d = queue.popleft()
        if seen[r][c]:                                    # đánh dấu muộn: bỏ qua nếu đã xử lý
            continue
        seen[r][c] = True
        if (r, c) == (n - 1, n - 1):
            return pushes
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                queue.append((nr, nc, d + 1))
                pushes += 1
    return pushes

for n in (5, 10, 20):
    print(f"lưới {n}x{n} toàn 0: đánh dấu muộn đưa vào hàng đợi {count_pushes_late(n)} lần; đánh dấu sớm chỉ tối đa {n * n} lần")

lưới 5x5 toàn 0: đánh dấu muộn đưa vào hàng đợi 73 lần; đánh dấu sớm chỉ tối đa 25 lần
lưới 10x10 toàn 0: đánh dấu muộn đưa vào hàng đợi 343 lần; đánh dấu sớm chỉ tối đa 100 lần
lưới 20x20 toàn 0: đánh dấu muộn đưa vào hàng đợi 1483 lần; đánh dấu sớm chỉ tối đa 400 lần


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (BFS một chiều): đơn giản, `O(n^2)`, đủ nhanh với `n <= 100`.

```python
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:                 # ô đầu hoặc ô cuối bị chặn
            return -1
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        queue = deque([(0, 0, 1)])                           # (hàng, cột, số ô đã đi)
        while queue:
            r, c, d = queue.popleft()
            if r == n - 1 and c == n - 1:                    # tới ô cuối: BFS đảm bảo d nhỏ nhất
                return d
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):                        # 8 hướng
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                        seen[nr][nc] = True                  # đánh dấu ngay khi đưa vào hàng đợi
                        queue.append((nr, nc, d + 1))
        return -1
```

In [16]:
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:                 # ô đầu hoặc ô cuối bị chặn
            return -1
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        queue = deque([(0, 0, 1)])                           # (hàng, cột, số ô đã đi)
        while queue:
            r, c, d = queue.popleft()
            if r == n - 1 and c == n - 1:                    # tới ô cuối: BFS đảm bảo d nhỏ nhất
                return d
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):                        # 8 hướng
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                        seen[nr][nc] = True                  # đánh dấu ngay khi đưa vào hàng đợi
                        queue.append((nr, nc, d + 1))
        return -1

sol = lambda g: Solution().shortestPathBinaryMatrix([row[:] for row in g])
for grid, expected in examples:
    got = sol(grid)
    print(got, "OK" if got == expected else "SAI")
    assert got == expected

2 OK
4 OK
-1 OK


## 10. Vì sao đúng

**Mô hình đồ thị.** Đỉnh là các ô có giá trị `0`; hai đỉnh nối nhau khi hai ô khác nhau và chung cạnh hoặc chung đỉnh. Độ dài của một đường đi sạch là số đỉnh của nó, bằng số cạnh cộng `1`. Vậy bài toán là tìm đường đi **ít cạnh nhất** từ `(0, 0)` đến `(n - 1, n - 1)` trong đồ thị không trọng số.

**BFS đúng.** Gọi `delta(v)` là số ô ít nhất của một đường đi sạch từ `(0, 0)` tới `v`. Khẳng định: ô nào được gán nhãn `d` thì `delta = d`, và các ô vào hàng đợi theo thứ tự nhãn không giảm. Quy nạp theo `d`. Cơ sở: `(0, 0)` có nhãn `1 = delta`. Bước quy nạp: giả sử đúng với mọi ô nhãn `<= d`.

- Mọi ô `v` có `delta(v) = d + 1` có ô liền trước `u` trên một đường ngắn nhất với `delta(u) = d`; theo giả thiết `u` được gán nhãn `d`, và khi `u` được lấy ra khỏi hàng đợi thì `v` (nếu chưa thăm) được gán nhãn `d + 1`.
- Ngược lại, ô `v` được gán nhãn `d + 1` từ một ô `u` nhãn `d` có `delta(v) <= d + 1`; nếu `delta(v) <= d` thì theo giả thiết quy nạp `v` đã được gán nhãn `delta(v)` từ trước nên đã thăm, không thể được gán nhãn `d + 1`. Vậy `delta(v) = d + 1`.

Do đó lần đầu tới ô cuối cho khoảng cách ngắn nhất; nếu hàng đợi cạn mà chưa tới thì không có đường đi: `-1`.

**Đánh dấu khi đưa vào.** Mỗi ô nhận nhãn đúng một lần (khi lần đầu được phát hiện, nhãn là nhỏ nhất); mỗi ô vào hàng đợi tối đa một lần nên tổng công việc `O(n^2)`.

**Trường hợp đặc biệt.** `grid[0][0] == 1` hoặc `grid[n - 1][n - 1] == 1`: không có đường đi sạch (ô đầu/ô cuối phải là `0`). `n = 1` và ô là `0`: ô đầu trùng ô cuối ⇒ độ dài `1` (nhờ kiểm tra đích khi **lấy ra** `(0, 0, 1)`).

**Cách 3 (hai chiều).** Hai phía mở rộng theo từng lớp nên `dist_a[u]` và `dist_b[v]` đều là khoảng cách ngắn nhất (tính bằng số ô) từ mỗi đầu tới ô đó. Mọi đường đi ngắn nhất `P` có một cạnh `u-v` nối một ô thuộc vùng đã thăm của phía này với ô thuộc vùng của phía kia; khi hai vùng lần đầu chạm nhau, độ dài ngắn nhất bằng `min(dist_a[u] + dist_b[v])` trên mọi cạnh như vậy trong lớp đang mở rộng, nên ta duyệt **trọn lớp** rồi lấy nhỏ nhất (không dừng ở lần gặp đầu tiên). **Cách 4 (A-star)**: heuristic Chebyshev không vượt quá khoảng cách thật và nhất quán (mỗi bước giảm `h` không quá `1`), nên A* với việc bỏ qua bản cũ trả về đường ngắn nhất. Hai cách này đều được kiểm bằng máy (bên dưới).

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi đường đi đơn (cho `n <= 3` đầy đủ, `n = 4` ngẫu nhiên). Ta thêm một oracle độc lập thứ hai dùng **thư giãn Bellman-Ford** trên cả lưới (không dùng hàng đợi), chạy cho `n` lớn hơn; rồi đối chiếu mọi cách.

In [17]:
def shortest_bellman_ford(grid: List[List[int]]) -> int:
    """Oracle thứ hai: thư giãn lặp lại dist[ô] = min(dist[lân cận] + 1) cho tới khi không đổi (không dùng hàng đợi)."""
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    INF = float("inf")
    dist = [[INF] * n for _ in range(n)]
    dist[0][0] = 1
    changed = True
    while changed:
        changed = False
        for r in range(n):
            for c in range(n):
                if grid[r][c]:
                    continue
                for dr, dc in dirs:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and dist[nr][nc] + 1 < dist[r][c]:
                        dist[r][c] = dist[nr][nc] + 1
                        changed = True
    return -1 if dist[n - 1][n - 1] == INF else dist[n - 1][n - 1]

impls = {"BFS": shortest_bfs, "hai chiều": shortest_bidirectional, "A*": shortest_astar,
         "nộp (Cách 2)": sol, "Cách 3 (nộp)": sol_bi, "Cách 4 (nộp)": sol_astar}

# (1) mọi lưới n <= 3 (2^1 + 2^4 + 2^9 lưới) với oracle liệt kê đường đi đơn và Bellman-Ford
total = 0
for n in (1, 2, 3):
    for bits in product((0, 1), repeat=n * n):
        grid = [list(bits[i * n:(i + 1) * n]) for i in range(n)]
        ref = shortest_brute(grid)
        assert shortest_bellman_ford(grid) == ref, grid
        for name, fn in impls.items():
            assert fn(grid) == ref, (name, grid)
        total += 1
print("đã đối chiếu", total, "lưới (n <= 3, mọi cấu hình): mọi cách khớp hai oracle")

đã đối chiếu 530 lưới (n <= 3, mọi cấu hình): mọi cách khớp hai oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường đi đơn | mũ | `O(n^2)` |
| 2. BFS | `O(n^2)` | `O(n^2)` |
| 3. BFS hai chiều | `O(n^2)` | `O(n^2)` |
| 4. A* | `O(n^2 log n)` | `O(n^2)` |

Đo ở cỡ tối đa (`n = 100`) với các dạng: lưới toàn `0`, ngẫu nhiên thưa, ngẫu nhiên dày, mê cung ngoằn ngoèo, không có đường:

In [18]:
def serpentine(n):
    """Mê cung ngoằn ngoèo: hàng lẻ là tường có một khe ở phía xen kẽ; đường đi rất dài."""
    grid = [[0] * n for _ in range(n)]
    for r in range(1, n, 2):
        for c in range(n):
            grid[r][c] = 1
        grid[r][n - 1 if (r // 2) % 2 == 0 else 0] = 0
    return grid

rng = random.Random(11)
n = 100
cases = {
    "toàn 0": [[0] * n for _ in range(n)],
    "ngẫu nhiên 20% chặn": [[1 if rng.random() < 0.2 and (r, c) not in ((0, 0), (n - 1, n - 1)) else 0 for c in range(n)] for r in range(n)],
    "ngẫu nhiên 40% chặn": [[1 if rng.random() < 0.4 and (r, c) not in ((0, 0), (n - 1, n - 1)) else 0 for c in range(n)] for r in range(n)],
    "mê cung ngoằn ngoèo (99x99)": serpentine(99),
    "không có đường (hàng tường kín)": [[0] * n for _ in range(n // 2)] + [[1] * n] + [[0] * n for _ in range(n - n // 2 - 1)],
    "đích bị chặn": [[0] * (n - 1) + [0] for _ in range(n - 1)] + [[0] * (n - 1) + [1]],
}
for label, grid in cases.items():
    ref = sol(grid)
    row = [f"{label:32}"]
    for name, fn in (("BFS", sol), ("hai chiều", sol_bi), ("A*", sol_astar)):
        t0 = time.perf_counter()
        got = fn(grid)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name, got, ref)
        row.append(f"{name} {dt:7.2f} ms")
    print(" | ".join(row), "| đáp án", ref)

toàn 0                           | BFS   17.69 ms | hai chiều   17.41 ms | A*   17.60 ms | đáp án 100


ngẫu nhiên 20% chặn              | BFS   14.47 ms | hai chiều   15.59 ms | A*   14.04 ms | đáp án 107
ngẫu nhiên 40% chặn              | BFS    0.08 ms | hai chiều    0.07 ms | A*    0.07 ms | đáp án -1


mê cung ngoằn ngoèo (99x99)      | BFS    8.47 ms | hai chiều    8.25 ms | A*    6.69 ms | đáp án 4804


không có đường (hàng tường kín)  | BFS    6.68 ms | hai chiều    6.63 ms | A*    6.42 ms | đáp án -1
đích bị chặn                     | BFS    0.02 ms | hai chiều    0.02 ms | A*    0.02 ms | đáp án -1


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed (đúng ràng buộc), test cỡ lớn và các tính chất. Mỗi lần gọi truyền bản sao của lưới.

In [19]:
def check_all(grid, expected=None, oracle="auto"):
    results = {name: fn([row[:] for row in grid]) for name, fn in impls.items()}
    first = results["nộp (Cách 2)"]
    for name, got in results.items():
        assert got == first, (name, got, first, grid if len(grid) <= 6 else len(grid))
    n = len(grid)
    if oracle == "auto":
        if n <= 3:
            assert shortest_brute([row[:] for row in grid]) == first, grid
        if n <= 12:
            assert shortest_bellman_ford([row[:] for row in grid]) == first, grid
    if expected is not None:
        assert first == expected, (grid, first, expected)
    return first

# 12.1 ví dụ đề
for g, e in examples:
    check_all(g, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[0]], 1),                                             # n = 1, ô là 0: đường đi gồm đúng một ô
    ([[1]], -1),                                            # n = 1, ô bị chặn
    ([[0, 0], [0, 0]], 2),                                  # lưới 2x2 trống: đi chéo
    ([[0, 1], [1, 0]], 2),                                  # chỉ đi được theo đường chéo
    ([[0, 1], [1, 1]], -1),                                 # ô cuối bị chặn
    ([[1, 0], [0, 0]], -1),                                 # ô đầu bị chặn
    ([[0, 0], [0, 1]], -1),                                 # ô cuối bị chặn (ô khác trống)
    ([[0] * 3 for _ in range(3)], 3),                       # lưới 3x3 trống: đi chéo, 3 ô
    ([[0, 0, 0], [1, 1, 0], [1, 1, 0]], 4),                 # ví dụ 2
    ([[0, 0, 1, 1], [1, 0, 1, 1], [1, 0, 0, 1], [1, 1, 0, 0]], 4),   # đường chéo chính trống: 4 ô
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 4),                 # ô giữa bị chặn nên không đi được đường chéo chính: (0,0),(0,1),(1,2),(2,2)
    ([[0, 1, 1], [1, 1, 1], [1, 1, 0]], -1),                # hai góc cô lập
    ([[0, 0, 0, 0], [1, 1, 1, 0], [0, 0, 0, 0], [0, 1, 1, 0]], 6),   # đường đi vòng ngoằn ngoèo
    ([[0] * 100 for _ in range(100)], 100),                 # lưới 100x100 trống: 100 ô theo đường chéo
]
for g, e in edge:
    check_all(g, e)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 14 ca


Ghi chú về ca `[[0,0,0],[0,1,0],[0,0,0]]`: đường chéo chính bị ô giữa chặn nên phải đi vòng: `(0,0) -> (0,1) -> (1,2) -> (2,2)` dài `4` ô. Ta kiểm lại bằng cả hai oracle:

In [20]:
g = [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
print("BFS:", sol(g), "| oracle đường đi đơn:", shortest_brute(g), "| Bellman-Ford:", shortest_bellman_ford(g))

BFS: 4 | oracle đường đi đơn: 4 | Bellman-Ford: 4


In [21]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_no_start_check(grid):                      # quên kiểm tra ô đầu/ô cuối là 1
    n = len(grid)
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
    return -1

def wrong_four_dirs(grid):                           # chỉ đi 4 hướng
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
    return -1

def wrong_count_edges(grid):                         # đếm số cạnh (số bước) thay vì số ô
    res = shortest_bfs(grid)
    return res - 1 if res > 0 else res

def wrong_dfs(grid):                                 # DFS: đường đầu tiên tìm được, không đảm bảo ngắn nhất
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    stack = [(0, 0, 1)]
    seen[0][0] = True
    while stack:
        r, c, d = stack.pop()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                stack.append((nr, nc, d + 1))
    return -1

def wrong_check_at_neighbor(grid):                   # kiểm tra đích lúc sinh lân cận: sai khi n = 1 (đích trùng đầu)
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                if (nr, nc) == (n - 1, n - 1):
                    return d + 1
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
    return -1

def random_grid(rng, n, p):
    return [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(n)]

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(1, 5)
        grid = random_grid(rng, n, rng.choice([0.2, 0.4, 0.6]))
        if best is not None and n >= len(best):
            continue
        if wrong([row[:] for row in grid]) != shortest_bellman_ford(grid):
            best = grid
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "quên kiểm tra ô đầu/ô cuối là 1": (wrong_no_start_check, [[1, 0, 0], [1, 1, 0], [1, 1, 0]]),               # ví dụ 3 của đề: ô đầu là 1
    "chỉ đi 4 hướng": (wrong_four_dirs, [[0, 1], [1, 0]]),                                                       # ví dụ 1 của đề
    "đếm số cạnh thay vì số ô": (wrong_count_edges, [[0, 0, 0], [1, 1, 0], [1, 1, 0]]),                         # ví dụ 2 của đề
    "dùng DFS thay cho BFS": (wrong_dfs, None),                                                                  # dựng bằng tìm kiếm ngay bên dưới
    "kiểm tra đích khi sinh lân cận (n = 1)": (wrong_check_at_neighbor, [[0]]),
}
hand_cases["dùng DFS thay cho BFS"] = (wrong_dfs, find_counter(wrong_dfs))        # phản ví dụ nhỏ nhất của DFS (tìm bằng tìm kiếm)
counters = {}
for name, (fn, grid) in hand_cases.items():
    wrong_result = fn([row[:] for row in grid])
    right = shortest_bellman_ford(grid)
    assert wrong_result != right, (name, wrong_result, right)
    assert find_counter(fn) is not None, name           # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = grid
    shown = grid if len(grid) <= 4 else f"{len(grid)}x{len(grid)}"
    print(f"{name:42} grid={shown}: sai -> {wrong_result}, đúng -> {right}")
    check_all(grid)
print("12.3 phản ví dụ: OK")

quên kiểm tra ô đầu/ô cuối là 1            grid=[[1, 0, 0], [1, 1, 0], [1, 1, 0]]: sai -> 4, đúng -> -1
chỉ đi 4 hướng                             grid=[[0, 1], [1, 0]]: sai -> -1, đúng -> 2
đếm số cạnh thay vì số ô                   grid=[[0, 0, 0], [1, 1, 0], [1, 1, 0]]: sai -> 3, đúng -> 4


dùng DFS thay cho BFS                      grid=[[0, 0, 1, 1], [0, 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 0]]: sai -> 6, đúng -> 5
kiểm tra đích khi sinh lân cận (n = 1)     grid=[[0]]: sai -> -1, đúng -> 1
12.3 phản ví dụ: OK


In [22]:
# 12.4 ngẫu nhiên nhỏ: nhiều mật độ chặn
rng = random.Random(4091)
for _ in range(3000):
    n = rng.randint(1, 9)
    check_all(random_grid(rng, n, rng.choice([0.0, 0.1, 0.3, 0.5, 0.7])))
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên vừa (so Bellman-Ford) và lớn (so các cách nhanh với nhau)
for _ in range(60):
    n = rng.randint(10, 25)
    grid = random_grid(rng, n, rng.choice([0.2, 0.35, 0.5]))
    grid[0][0] = grid[n - 1][n - 1] = 0
    ref = shortest_bellman_ford(grid)
    assert check_all(grid, oracle="skip") == ref
for _ in range(15):
    n = rng.randint(50, 100)
    grid = random_grid(rng, n, rng.choice([0.1, 0.3, 0.4]))
    grid[0][0] = grid[n - 1][n - 1] = 0
    check_all(grid, oracle="skip")
print("12.5 ngẫu nhiên vừa và lớn: OK")

# 12.6 tính chất
def path_lower_bound(n):
    return n                                         # đi chéo cần ít nhất n ô

for _ in range(400):
    n = rng.randint(1, 9)
    grid = random_grid(rng, n, rng.choice([0.1, 0.3, 0.5]))
    res = sol(grid)
    assert res == -1 or path_lower_bound(n) <= res <= n * n                           # n <= độ dài <= số ô
    if res != -1:
        # mở thêm một ô chặn thành trống không làm đường đi dài hơn
        r, c = rng.randrange(n), rng.randrange(n)
        g2 = [row[:] for row in grid]
        g2[r][c] = 0
        assert sol(g2) <= res
        # lấp thêm một ô trống thành chặn không làm đường đi ngắn hơn
        g3 = [row[:] for row in grid]
        g3[r][c] = 1
        res3 = sol(g3)
        assert res3 == -1 or res3 >= res
    assert sol([list(col) for col in zip(*grid)]) == res                              # chuyển vị (đối xứng qua đường chéo chính) không đổi
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK


12.5 ngẫu nhiên vừa và lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra ô đầu / ô cuối là `1`

Đường đi sạch phải đi qua toàn ô `0`, kể cả ô đầu và ô cuối.

In [23]:
grid = counters["quên kiểm tra ô đầu/ô cuối là 1"]
print("grid =", grid, "| sai ->", wrong_no_start_check(grid), "| đúng ->", sol(grid))
assert wrong_no_start_check(grid) != sol(grid)
print("lưới [[1]]:", wrong_no_start_check([[1]]), "(đúng là -1)")

grid = [[1, 0, 0], [1, 1, 0], [1, 1, 0]] | sai -> 4 | đúng -> -1
lưới [[1]]: 1 (đúng là -1)


Vì sao sai: BFS bắt đầu từ `(0, 0)` mà không kiểm tra ô đó là `1` vẫn lan ra các ô khác. **Đúng:** `if grid[0][0] or grid[n - 1][n - 1]: return -1`.

### Lỗi 2 — Chỉ đi `4` hướng

Đề cho phép đi **chéo** (kề chung đỉnh).

In [24]:
grid = counters["chỉ đi 4 hướng"]
print("grid =", grid, "| sai ->", wrong_four_dirs(grid), "| đúng ->", sol(grid))
assert wrong_four_dirs(grid) != sol(grid)

grid = [[0, 1], [1, 0]] | sai -> -1 | đúng -> 2


Vì sao sai: `[[0,1],[1,0]]` chỉ nối được bằng đường chéo. **Đúng:** `8` hướng: `dr, dc` trong `(-1, 0, 1)` trừ `(0, 0)`.

### Lỗi 3 — Đếm số cạnh thay vì số ô

Đề định nghĩa độ dài là **số ô** đi qua (cả ô đầu và ô cuối), nên bằng số cạnh cộng `1`.

In [25]:
grid = counters["đếm số cạnh thay vì số ô"]
print("grid =", grid, "| sai ->", wrong_count_edges(grid), "| đúng ->", sol(grid))
assert wrong_count_edges(grid) != sol(grid)

grid = [[0, 0, 0], [1, 1, 0], [1, 1, 0]] | sai -> 3 | đúng -> 4


Vì sao sai: lệch một đơn vị. **Đúng:** khởi tạo độ dài của ô đầu là `1`.

### Lỗi 4 — Dùng DFS thay cho BFS

DFS trả về **một** đường đi, không đảm bảo ngắn nhất.

In [26]:
grid = counters["dùng DFS thay cho BFS"]
print("grid =", grid, "| sai (DFS) ->", wrong_dfs([row[:] for row in grid]), "| đúng ->", sol(grid))
assert wrong_dfs([row[:] for row in grid]) != sol(grid)

grid = [[0, 0, 1, 1], [0, 0, 0, 0], [0, 1, 1, 0], [0, 0, 0, 0]] | sai (DFS) -> 6 | đúng -> 5


Vì sao sai: DFS đi sâu theo một hướng rồi mới quay lại, đường tìm thấy đầu tiên thường dài. **Đúng:** BFS mở rộng theo từng lớp khoảng cách.

### Lỗi 5 — Kiểm tra đích khi sinh lân cận (hỏng `n = 1`)

Nếu chỉ kiểm tra "đã tới đích" khi xét **lân cận**, ô đầu trùng ô cuối (`n = 1`) không bao giờ được kiểm tra.

In [27]:
grid = counters["kiểm tra đích khi sinh lân cận (n = 1)"]
print("grid =", grid, "| sai ->", wrong_check_at_neighbor(grid), "| đúng ->", sol(grid))
assert wrong_check_at_neighbor(grid) != sol(grid)

grid = [[0]] | sai -> -1 | đúng -> 1


Vì sao sai: với `[[0]]` đáp án là `1` nhưng cách này trả về `-1`. **Đúng:** kiểm tra đích khi **lấy ra** khỏi hàng đợi (hoặc xử lý riêng `n = 1`).

### Lỗi 6 — Đánh dấu đã thăm muộn

Đánh dấu khi lấy ra làm một ô bị đưa vào hàng đợi rất nhiều lần (xem mục 8.3).

In [28]:
for n in (10, 20):
    print(f"lưới {n}x{n} toàn 0: đánh dấu muộn -> {count_pushes_late(n)} lần đưa vào, đánh dấu sớm -> tối đa {n * n} lần")
assert count_pushes_late(20) > 20 * 20

lưới 10x10 toàn 0: đánh dấu muộn -> 343 lần đưa vào, đánh dấu sớm -> tối đa 100 lần
lưới 20x20 toàn 0: đánh dấu muộn -> 1483 lần đưa vào, đánh dấu sớm -> tối đa 400 lần


Vì sao chậm: nhiều ô cùng lúc nằm trong hàng đợi và phát sinh thêm nhiều bản sao. **Đúng:** đánh dấu ngay khi `append` vào hàng đợi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đường đi ngắn nhất, mỗi bước chi phí bằng nhau, trên lưới / đồ thị không trọng số" ⇒ **BFS**.
- Lưu `(r, c, d)` trong hàng đợi (hoặc BFS theo lớp) để có độ dài khi tới đích.
- **Đánh dấu khi đưa vào** hàng đợi, kiểm tra đích khi **lấy ra** (đúng cả `n = 1`).
- Tạo danh sách `8` hướng một lần: `[(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if dr or dc]`.
- Có thể tăng tốc bằng **BFS hai chiều** hoặc **A-star** với heuristic Chebyshev (di chuyển `8` hướng).
- Cùng họ: Shortest Path in Binary Matrix (đúng bài này), Rotting Oranges (994), 01 Matrix (542), Shortest Path with Obstacles Elimination (1293), Word Ladder (127).
- Test nên có: `n = 1` (`0` và `1`), ô đầu/ô cuối bị chặn, đường đi chỉ chéo, mê cung ngoằn ngoèo, không có đường, `n = 100`.

## 15. Biến thể

1. **Chỉ đi `4` hướng** (mê cung thường): cùng BFS với `4` hướng.
2. **Đếm số đường đi ngắn nhất:** BFS ghi thêm số cách tới mỗi ô.
3. **Có thể phá tối đa `k` chướng ngại vật (1293):** BFS trên trạng thái `(r, c, số vật đã phá)`.
4. Bài liên quan: 542, 994, 1293.

Kiểm biến thể 2 (đếm số đường đi ngắn nhất, 8 hướng) bằng oracle liệt kê mọi đường đi đơn:

In [29]:
def count_shortest_paths(grid: List[List[int]]):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1, 0
    dist = [[0] * n for _ in range(n)]
    ways = [[0] * n for _ in range(n)]
    dist[0][0], ways[0][0] = 1, 1
    queue = deque([(0, 0)])
    while queue:
        r, c = queue.popleft()
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc]:
                if not dist[nr][nc]:
                    dist[nr][nc] = dist[r][c] + 1
                    queue.append((nr, nc))
                if dist[nr][nc] == dist[r][c] + 1:
                    ways[nr][nc] += ways[r][c]
    return (dist[n - 1][n - 1], ways[n - 1][n - 1]) if dist[n - 1][n - 1] else (-1, 0)

def count_shortest_paths_brute(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1, 0
    best, count = [float("inf")], [0]
    on_path = [[False] * n for _ in range(n)]
    def dfs(r, c, length):
        if length > best[0]:
            return
        if (r, c) == (n - 1, n - 1):
            if length < best[0]:
                best[0], count[0] = length, 1
            else:
                count[0] += 1
            return
        on_path[r][c] = True
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not on_path[nr][nc]:
                dfs(nr, nc, length + 1)
        on_path[r][c] = False
    dfs(0, 0, 1)
    return (best[0], count[0]) if best[0] != float("inf") else (-1, 0)

rng = random.Random(15)
for _ in range(600):
    n = rng.randint(1, 4)
    grid = random_grid(rng, n, rng.choice([0.0, 0.2, 0.4]))
    assert count_shortest_paths(grid) == count_shortest_paths_brute(grid), grid
assert count_shortest_paths([[0, 0], [0, 0]]) == (2, 1)
assert count_shortest_paths([[0, 0, 0], [0, 0, 0], [0, 0, 0]]) == (3, 1)
print("biến thể đếm số đường đi ngắn nhất: khớp oracle; lưới 4x4 trống:", count_shortest_paths([[0] * 4 for _ in range(4)]))

biến thể đếm số đường đi ngắn nhất: khớp oracle; lưới 4x4 trống: (4, 1)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi đơn | Theo định nghĩa | mũ | `O(n^2)` | Oracle, `n <= 4` |
| 2. BFS | Mở rộng theo lớp | `O(n^2)` | `O(n^2)` | **Cách nộp chính** |
| 3. BFS hai chiều | Gặp nhau ở giữa | `O(n^2)` | `O(n^2)` | Lưới lớn, thoáng |
| 4. A* | Heuristic Chebyshev | `O(n^2 log n)` | `O(n^2)` | Khi có heuristic tốt, một đích |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(len(dirs))
print(Solution().shortestPathBinaryMatrix([[0, 0], [0, 0]]), Solution().shortestPathBinaryMatrix([[0]]), Solution().shortestPathBinaryMatrix([[0, 1], [1, 1]]))
```

In [30]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # số hướng đi
my_guess_2 = None      # ba kết quả

### Bài 2 — Tính tay (dễ-vừa)

`grid = [[0,0,1,0],[1,0,1,0],[0,0,0,1],[0,1,0,0]]` (lưới `4 x 4` ở mục 8.2). Liệt kê một đường đi ngắn nhất và ghi độ dài.

In [31]:
# Bài 2: điền một đường đi ngắn nhất (danh sách ô) và độ dài
hand_path, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS.

In [32]:
def exercise_3(grid):
    n = len(grid)
    # TODO 1: nếu grid[0][0] hoặc grid[n - 1][n - 1] bằng 1 thì trả về -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        # TODO 2: nếu (r, c) là ô cuối thì trả về d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            # TODO 3: nếu ô (nr, nc) trong lưới, bằng 0 và chưa thăm: đánh dấu thăm rồi đưa (nr, nc, d + 1) vào hàng đợi
            pass
    return -1


# for g, e in examples:
#     assert exercise_3([row[:] for row in g]) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(grid)` bằng **BFS theo lớp** (một danh sách `frontier` cho mỗi lớp, tăng biến độ dài sau mỗi lớp).

In [33]:
def exercise_4(grid):
    # TODO
    pass


# for g, e in examples:
#     assert exercise_4([row[:] for row in g]) == e

### Bài 5 — Mini challenge: in ra một đường đi ngắn nhất (khó)

Viết `exercise_5(grid)` trả về **danh sách các ô** `[(0, 0), ..., (n - 1, n - 1)]` của **một** đường đi sạch ngắn nhất, hoặc `None` nếu không có. Có thể có nhiều đáp án đúng nên kiểm bằng hàm `check_path`.

Gợi ý: BFS ghi lại `parent[ô]` khi đưa vào hàng đợi, rồi truy ngược từ ô cuối.

In [34]:
def exercise_5(grid):
    # TODO
    pass


def check_path(grid, path) -> bool:
    n = len(grid)
    expected = shortest_bellman_ford(grid)
    if expected == -1:
        return path is None
    if path is None or len(path) != expected or path[0] != (0, 0) or path[-1] != (n - 1, n - 1):
        return False
    if any(grid[r][c] for r, c in path) or len(set(path)) != len(path):
        return False
    return all(max(abs(a[0] - b[0]), abs(a[1] - b[1])) == 1 for a, b in zip(path, path[1:]))


# for g, e in examples:
#     assert check_path(g, exercise_5([row[:] for row in g]))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `8` hướng gồm cả đường chéo.
- Mức 2: lưới `2 x 2` trống: đi chéo từ `(0, 0)` đến `(1, 1)`; `[[0]]`: ô đầu trùng ô cuối; `[[0,1],[1,1]]` ô cuối bị chặn.
- Mức 3: `8`; `2 1 -1`.

**Bài 2**
- Mức 1: tìm đường từ `(0, 0)` đến `(3, 3)`; ô `(3, 3)` là `0`.
- Mức 2: `(0, 0) -> (1, 1) -> (2, 2) -> (3, 3)` toàn là `0`? kiểm tra `grid[1][1] = 0`, `grid[2][2] = 0`, `grid[3][3] = 0`.
- Mức 3: đường đi `[(0,0), (1,1), (2,2), (3,3)]`, độ dài `4`.

**Bài 3**
- Mức 1: `if grid[0][0] or grid[n - 1][n - 1]: return -1`.
- Mức 2: `if r == n - 1 and c == n - 1: return d`.
- Mức 3: `if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]: seen[nr][nc] = True; queue.append((nr, nc, d + 1))`.

**Bài 4**
- Mức 1: `frontier = [(0, 0)]`, `d = 1`, `seen`.
- Mức 2: mỗi lớp: nếu `(n - 1, n - 1)` thuộc `frontier` trả về `d`; ngược lại tạo `next_frontier` từ các lân cận hợp lệ chưa thăm; `d += 1`.
- Mức 3: kết thúc khi `frontier` rỗng: trả về `-1`.

**Bài 5**
- Mức 1: `parent = {(0, 0): None}`.
- Mức 2: khi đưa `(nr, nc)` vào hàng đợi, ghi `parent[(nr, nc)] = (r, c)`.
- Mức 3: từ `(n - 1, n - 1)` đi ngược theo `parent` tới `(0, 0)`, đảo danh sách.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [35]:
r1 = len(dirs)
r2 = (Solution().shortestPathBinaryMatrix([[0, 0], [0, 0]]), Solution().shortestPathBinaryMatrix([[0]]), Solution().shortestPathBinaryMatrix([[0, 1], [1, 1]]))
print(r1, r2)
assert r1 == 8
assert r2 == (2, 1, -1)

8 (2, 1, -1)


### Đáp án Bài 2

`(0,0) -> (1,1) -> (2,2) -> (3,3)`: các ô `grid[0][0] = 0`, `grid[1][1] = 0`, `grid[2][2] = 0`, `grid[3][3] = 0`; độ dài `4` (không thể ngắn hơn vì cần ít nhất `n = 4` ô).

In [36]:
hand_path_solution, hand_answer_solution = [(0, 0), (1, 1), (2, 2), (3, 3)], 4
g = [[0, 0, 1, 0], [1, 0, 1, 0], [0, 0, 0, 1], [0, 1, 0, 0]]
assert all(g[r][c] == 0 for r, c in hand_path_solution)
assert len(hand_path_solution) == hand_answer_solution == sol(g) == shortest_bellman_ford(g)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [37]:
def exercise_3_solution(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    queue = deque([(0, 0, 1)])
    while queue:
        r, c, d = queue.popleft()
        if r == n - 1 and c == n - 1:
            return d
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                seen[nr][nc] = True
                queue.append((nr, nc, d + 1))
    return -1

for g, e in examples:
    assert exercise_3_solution([row[:] for row in g]) == e
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 7)
    g = random_grid(rng, n, rng.choice([0.1, 0.3, 0.5]))
    assert exercise_3_solution([row[:] for row in g]) == shortest_bellman_ford(g)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [38]:
def exercise_4_solution(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    seen = [[False] * n for _ in range(n)]
    seen[0][0] = True
    frontier, d = [(0, 0)], 1
    while frontier:
        if (n - 1, n - 1) in frontier:
            return d
        nxt = []
        for r, c in frontier:
            for dr, dc in dirs:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                    seen[nr][nc] = True
                    nxt.append((nr, nc))
        frontier, d = nxt, d + 1
    return -1

for g, e in examples:
    assert exercise_4_solution([row[:] for row in g]) == e
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 7)
    g = random_grid(rng, n, rng.choice([0.1, 0.3, 0.5]))
    assert exercise_4_solution([row[:] for row in g]) == shortest_bellman_ford(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
def exercise_5_solution(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return None
    parent = {(0, 0): None}
    queue = deque([(0, 0)])
    while queue:
        r, c = queue.popleft()
        if (r, c) == (n - 1, n - 1):
            path, cur = [], (r, c)
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and (nr, nc) not in parent:
                parent[(nr, nc)] = (r, c)
                queue.append((nr, nc))
    return None

def check_path_solution(grid, path):
    n = len(grid)
    expected = shortest_bellman_ford(grid)
    if expected == -1:
        return path is None
    if path is None or len(path) != expected or path[0] != (0, 0) or path[-1] != (n - 1, n - 1):
        return False
    if any(grid[r][c] for r, c in path) or len(set(path)) != len(path):
        return False
    return all(max(abs(a[0] - b[0]), abs(a[1] - b[1])) == 1 for a, b in zip(path, path[1:]))

for g, e in examples:
    assert check_path_solution(g, exercise_5_solution([row[:] for row in g]))
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 7)
    g = random_grid(rng, n, rng.choice([0.1, 0.3, 0.5]))
    assert check_path_solution(g, exercise_5_solution([row[:] for row in g])), g
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (BFS 8 hướng)

```python
class Solution:
    def shortestPathBinaryMatrix(self, grid: List[List[int]]) -> int:
        from collections import deque
        n = len(grid)
        if grid[0][0] or grid[n - 1][n - 1]:
            return -1
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        queue = deque([(0, 0, 1)])                    # (hàng, cột, số ô)
        while queue:
            r, c, d = queue.popleft()
            if r == n - 1 and c == n - 1:
                return d
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                        seen[nr][nc] = True           # đánh dấu khi đưa vào
                        queue.append((nr, nc, d + 1))
        return -1
```

Công thức nhớ nhanh: **kiểm tra ô đầu/cuối; BFS 8 hướng; độ dài = số ô (bắt đầu từ `1`); đánh dấu khi đưa vào; kiểm tra đích khi lấy ra**.

### Checklist trước khi submit

- [ ] Kiểm tra `grid[0][0]` và `grid[n - 1][n - 1]` bằng `1` thì trả `-1`?
- [ ] Đủ `8` hướng (kể cả đường chéo)?
- [ ] Độ dài tính bằng số ô, ô đầu có độ dài `1`?
- [ ] Đánh dấu đã thăm **khi đưa vào** hàng đợi?
- [ ] Kiểm tra đích khi **lấy ra** (đúng với `n = 1`)?
- [ ] Đã thử: `n = 1` (`0` và `1`), đường chỉ đi chéo, không có đường, mê cung ngoằn ngoèo, `n = 100`?